# 06 — Bed Savings (RQ4)

**Research Question:** How many beds can be freed?

Hospital beds are a scarce resource. Every unnecessary bed-day displaces another patient.
We quantify savings from three independent levers.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from shared import (
    load_kidney, load_hospital_tags, setup_plot_style,
    save_plot, save_metrics,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
print(f"Loaded {len(kidney):,} admissions, {len(tags)} hospitals")

total_bed_days = kidney["DIAS_PERM"].sum()
print(f"Total bed-days: {total_bed_days:,}")

Loaded 206,500 admissions, 510 hospitals
Total bed-days: 507,465


## Lever 1: Long-Stay Reduction

If patients staying >7 days had their stay reduced to 7 days, how many bed-days are saved?

In [2]:
LONG_STAY_THRESHOLD = 7

long_stay = kidney[kidney["DIAS_PERM"] > LONG_STAY_THRESHOLD].copy()
long_stay_n = len(long_stay)
long_stay_pct = long_stay_n / len(kidney) * 100
long_stay_bed_days = long_stay["DIAS_PERM"].sum()
long_stay_bed_pct = long_stay_bed_days / total_bed_days * 100

# Savings if capped at threshold
savings_lever1 = (long_stay["DIAS_PERM"] - LONG_STAY_THRESHOLD).sum()

print(f"Long-stay patients (>{LONG_STAY_THRESHOLD}d): {long_stay_n:,} ({long_stay_pct:.1f}%)")
print(f"Long-stay bed-days: {long_stay_bed_days:,} ({long_stay_bed_pct:.1f}% of total)")
print(f"Lever 1 savings: {savings_lever1:,} bed-days ({savings_lever1/total_bed_days*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins = list(range(0, 31)) + [100]
axes[0].hist(kidney["DIAS_PERM"].clip(upper=30), bins=30, color="#2563EB",
             edgecolor="white", alpha=0.8)
axes[0].axvline(LONG_STAY_THRESHOLD, color="red", linestyle="--",
                label=f"Threshold: {LONG_STAY_THRESHOLD}d")
axes[0].set_title("LOS Distribution")
axes[0].set_xlabel("Days")
axes[0].legend()

labels = [f"Long-stay patients\n({long_stay_pct:.1f}%)",
          f"Long-stay bed-days\n({long_stay_bed_pct:.1f}%)",
          f"Long-stay deaths\n({long_stay['MORTE'].sum()/max(kidney['MORTE'].sum(),1)*100:.1f}%)"]
values = [long_stay_pct, long_stay_bed_pct,
          long_stay["MORTE"].sum() / max(kidney["MORTE"].sum(), 1) * 100]
colors = ["#2563EB", "#D97706", "#DC2626"]
axes[1].bar(labels, values, color=colors)
axes[1].set_title("Long-Stay Pareto Effect")
axes[1].set_ylabel("%")

fig.suptitle("Lever 1: Long-Stay Reduction", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "lever1_long_stay", prefix="06")

Long-stay patients (>7d): 9,330 (4.5%)
Long-stay bed-days: 119,298 (23.5% of total)
Lever 1 savings: 53,988 bed-days (10.6%)
  Saved: 06_lever1_long_stay.png


## Lever 2: Diagnostic Outpatient Shift

If diagnostic admissions (imaging) were handled in outpatient settings, those bed-days are freed entirely.

In [3]:
diag = kidney[kidney["proc_category"] == "DIAGNOSTIC"].copy()
diag_bed_days = diag["DIAS_PERM"].sum()
savings_lever2 = diag_bed_days

print(f"Diagnostic admissions: {len(diag):,} ({len(diag)/len(kidney)*100:.1f}%)")
print(f"Diagnostic bed-days: {diag_bed_days:,} ({diag_bed_days/total_bed_days*100:.1f}%)")
print(f"Lever 2 savings: {savings_lever2:,} bed-days ({savings_lever2/total_bed_days*100:.1f}%)")

# Diagnostic stays by hospital
diag_by_hosp = diag.groupby("CNES").agg(
    n_diag=pd.NamedAgg(column="CNES", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
).reset_index()
diag_by_hosp = diag_by_hosp.merge(
    kidney.groupby("CNES").size().rename("n_total").reset_index(),
    on="CNES", how="left"
)
diag_by_hosp["diag_pct"] = diag_by_hosp["n_diag"] / diag_by_hosp["n_total"] * 100

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(diag_by_hosp["diag_pct"], bins=30, color="#DC2626", edgecolor="white", alpha=0.8)
ax.set_title("Hospital-Level: % of Admissions That Are Diagnostic")
ax.set_xlabel("% diagnostic admissions")
ax.set_ylabel("Number of hospitals")
fig.tight_layout()
save_plot(fig, "lever2_diagnostic_shift", prefix="06")

Diagnostic admissions: 41,487 (20.1%)
Diagnostic bed-days: 111,506 (22.0%)
Lever 2 savings: 111,506 bed-days (22.0%)


  Saved: 06_lever2_diagnostic_shift.png


## Lever 3: Hospital Standardization

If hospitals with above-median LOS (within their peer group) matched the group median.

In [4]:
# Compute peer-group median LOS
kidney_with_tags = kidney.merge(tags[["CNES", "comparability_group"]], on="CNES", how="left")
group_medians = kidney_with_tags.groupby("comparability_group")["DIAS_PERM"].median().rename("group_median_los")

kidney_with_tags = kidney_with_tags.merge(group_medians, on="comparability_group", how="left")
kidney_with_tags["excess_vs_group"] = (
    kidney_with_tags["DIAS_PERM"] - kidney_with_tags["group_median_los"]
).clip(lower=0)

savings_lever3 = kidney_with_tags["excess_vs_group"].sum()

print(f"Lever 3 savings: {savings_lever3:,.0f} bed-days ({savings_lever3/total_bed_days*100:.1f}%)")

Lever 3 savings: 218,232 bed-days (43.0%)


## Combined Scenario

In [5]:
levers = {
    "Long-stay\nreduction": savings_lever1,
    "Diagnostic\noutpatient shift": savings_lever2,
    "Hospital\nstandardization": savings_lever3,
}

# Conservative combined (not additive — some overlap)
combined_max = sum(levers.values())
combined_conservative = combined_max * 0.7  # 30% overlap estimate

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Waterfall
names = list(levers.keys())
values = list(levers.values())
colors = ["#2563EB", "#D97706", "#059669"]
axes[0].bar(names, values, color=colors)
axes[0].set_title("Bed-Day Savings by Lever")
axes[0].set_ylabel("Bed-days saved")
for i, v in enumerate(values):
    axes[0].text(i, v + total_bed_days * 0.005,
                f"{v:,.0f}\n({v/total_bed_days*100:.1f}%)", ha="center", fontsize=9)

# Summary
summary_data = {
    "Total bed-days": total_bed_days,
    "Combined\n(optimistic)": combined_max,
    "Combined\n(conservative)": combined_conservative,
}
axes[1].bar(summary_data.keys(), summary_data.values(),
           color=["#6B7280", "#7C3AED", "#059669"])
axes[1].set_title("Total vs Saveable Bed-Days")
axes[1].set_ylabel("Bed-days")
for i, (k, v) in enumerate(summary_data.items()):
    axes[1].text(i, v + total_bed_days * 0.005, f"{v:,.0f}", ha="center", fontsize=9)

fig.suptitle("Bed-Day Savings — Three Levers", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "combined_savings", prefix="06")

  Saved: 06_combined_savings.png


## Summary Metrics

In [6]:
metrics = {
    "total_bed_days": int(total_bed_days),
    "lever1_long_stay_savings": int(savings_lever1),
    "lever2_diagnostic_shift_savings": int(savings_lever2),
    "lever3_standardization_savings": int(savings_lever3),
    "combined_optimistic": int(combined_max),
    "combined_conservative": int(combined_conservative),
    "combined_pct_optimistic": round(combined_max / total_bed_days * 100, 1),
    "combined_pct_conservative": round(combined_conservative / total_bed_days * 100, 1),
    "long_stay_patients_pct": round(long_stay_pct, 1),
    "diagnostic_admissions_pct": round(len(diag) / len(kidney) * 100, 1),
}
save_metrics(metrics, "06_bed_savings")

print("\n=== BED SAVINGS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 06_bed_savings.json

=== BED SAVINGS ===
  total_bed_days: 507465
  lever1_long_stay_savings: 53988
  lever2_diagnostic_shift_savings: 111506
  lever3_standardization_savings: 218232
  combined_optimistic: 383726
  combined_conservative: 268608
  combined_pct_optimistic: 75.6
  combined_pct_conservative: 52.9
  long_stay_patients_pct: 4.5
  diagnostic_admissions_pct: 20.1
